### Architecture-

**Driver** (brain of the operation): It runs your main PySpark code, creates the SparkSession, and decides how to split up the work.
It also coordinates all tasks and collects the final results from workers.

**Cluster Manager:** The resource allocator grabs necessary RAM and CPU power.
It manages available resources across the cluster and assigns them to Spark applications.

**Executors:** These are the workers that execute the tasks assigned by the Driver and store data in their memory.
They perform computations in parallel and send results back to the Driver.


### Lazy Evaluation and DAG

**DAG (Directed Acyclic Graph):**
When you apply Transformations (like filter(), withColumn()), Apache Spark does not execute them immediately. Instead, it builds a DAG, which is a step-by-step blueprint (or lineage) of all operations you want to perform.

**Action triggers execution:**
Spark executes this blueprint only when you call an Action (like show(), count(), collect()). This lets Spark analyze the entire DAG, optimize the execution plan, reduce unnecessary computations, and improve performance before doing actual work.

# Pipeline

In [12]:
!pip install pyspark

## Session Building

In [13]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark_Assignment") \
    .getOrCreate()

print("Spark Session Created Sussesfully!")

Spark Session Created Sussesfully!


In [14]:
df = spark.read.csv( "/content/retail_sales_dataset.csv", header=True, inferSchema=True)
print("Data loaded into spark session")

Data loaded into spark session


In [15]:
#printing 5 rowss of dataset
df.show(5)

+--------------+----------+-----------+------+---+----------------+--------+--------------+------------+
|Transaction ID|      Date|Customer ID|Gender|Age|Product Category|Quantity|Price per Unit|Total Amount|
+--------------+----------+-----------+------+---+----------------+--------+--------------+------------+
|             1|2023-11-24|    CUST001|  Male| 34|          Beauty|       3|            50|         150|
|             2|2023-02-27|    CUST002|Female| 26|        Clothing|       2|           500|        1000|
|             3|2023-01-13|    CUST003|  Male| 50|     Electronics|       1|            30|          30|
|             4|2023-05-21|    CUST004|  Male| 37|        Clothing|       1|           500|         500|
|             5|2023-05-06|    CUST005|  Male| 30|          Beauty|       2|            50|         100|
+--------------+----------+-----------+------+---+----------------+--------+--------------+------------+
only showing top 5 rows


In [16]:
# printing Schema
print("Data Schema (Column Types)")
df.printSchema()

Data Schema (Column Types)
root
 |-- Transaction ID: integer (nullable = true)
 |-- Date: date (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Price per Unit: integer (nullable = true)
 |-- Total Amount: integer (nullable = true)



## Modify & Transform

In [17]:
from pyspark.sql.functions import col, round

# Keep only required columns
selected_df = df.select(
    "Transaction ID",
    "Customer ID",
    "Age",
    "Quantity",
    "Price per Unit",
    "Total Amount"
)

# Rename column
transformed_df = selected_df \
    .withColumnRenamed("Transaction ID", "transaction_id") \
    .withColumnRenamed("Customer ID", "customer_id") \
    .withColumnRenamed("Price per Unit", "price_per_unit") \
    .withColumnRenamed("Total Amount", "total_amount")

# Cast numeric columns
transformed_df = transformed_df \
    .withColumn("price_per_unit", col("price_per_unit").cast("double")) \
    .withColumn("Quantity", col("Quantity").cast("integer"))

# Calculate final price
transformed_df = transformed_df.withColumn(
    "calculated_total_amount",
    round(col("Quantity") * col("price_per_unit"), 2)
)

print("Transformed DataFrame")
transformed_df.show(5)

print("New Schema")
transformed_df.printSchema()

Transformed DataFrame
+--------------+-----------+---+--------+--------------+------------+-----------------------+
|transaction_id|customer_id|Age|Quantity|price_per_unit|total_amount|calculated_total_amount|
+--------------+-----------+---+--------+--------------+------------+-----------------------+
|             1|    CUST001| 34|       3|          50.0|         150|                  150.0|
|             2|    CUST002| 26|       2|         500.0|        1000|                 1000.0|
|             3|    CUST003| 50|       1|          30.0|          30|                   30.0|
|             4|    CUST004| 37|       1|         500.0|         500|                  500.0|
|             5|    CUST005| 30|       2|          50.0|         100|                  100.0|
+--------------+-----------+---+--------+--------------+------------+-----------------------+
only showing top 5 rows
New Schema
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |

## Null Values and Filter Data

In [18]:
# dropping all null value records
cleaned_df = transformed_df.dropna(subset=["customer_id", "calculated_total_amount"])

# filtering data on the bases of age and quantity
filtered_df = cleaned_df \
    .filter(col("Age") >= 25) \
    .filter(col("Quantity") > 1)


print("Cleaned & Filtered Data")
filtered_df.show(5)

print(f"Total rows after filtering: {filtered_df.count()}")



Cleaned & Filtered Data
+--------------+-----------+---+--------+--------------+------------+-----------------------+
|transaction_id|customer_id|Age|Quantity|price_per_unit|total_amount|calculated_total_amount|
+--------------+-----------+---+--------+--------------+------------+-----------------------+
|             1|    CUST001| 34|       3|          50.0|         150|                  150.0|
|             2|    CUST002| 26|       2|         500.0|        1000|                 1000.0|
|             5|    CUST005| 30|       2|          50.0|         100|                  100.0|
|             7|    CUST007| 46|       2|          25.0|          50|                   50.0|
|             8|    CUST008| 30|       4|          25.0|         100|                  100.0|
+--------------+-----------+---+--------+--------------+------------+-----------------------+
only showing top 5 rows
Total rows after filtering: 638


In [19]:
#  Save the processed data as csv and Parquet file
filtered_df.write.mode("overwrite").csv("/content/retail_sales_cleaned_csv", header=True )

filtered_df.write.mode("overwrite").parquet( "/content/retail_sales_cleaned_parquet" )

print("files saved in both formats.")

files saved in both formats.


**Performance & Best Practices for normal size data set**

**CSV (Row-based):** Good for human readability, but not ideal for big data performance. It takes more storage space, is slower to read/write, and loses schema information when saved.
Since data is stored row by row, Spark must scan the entire file even if only a few columns are needed.

**Parquet (Column-based): **This is the preferred format in Apache Spark. It compresses data efficiently, preserves the exact schema, and supports Predicate Pushdown (if you query only one column, Spark reads only that column).
This reduces disk I/O significantly and makes analytics queries much faster.

**Best Practices for Large Datasets**

**Avoid collect(): **This command tries to bring all data from Executor/Worker nodes to the Driver node’s memory. On large datasets, this can cause memory overflow and crash the Spark application.
Use it only for very small datasets where the data can safely fit into Driver memory.

**Use show():** Throughout the pipeline, I used show(5) to safely preview the data without overwhelming the Driver.
It displays only a limited number of rows, making debugging and inspection much safer and faster in Apache Spark.